# Notebook 1: Extract and Load Raw Data
This notebook handles the extraction of raw data from a CSV file and loads it into the `raw_survey` table in a PostgreSQL database. It includes the following steps:
- Setting up a secure database connection using environment variables.
- Executing the `setup.py` script to create the database and table if they don’t exist.
- Loading the CSV data into a Pandas DataFrame.
- Checking if data already exists in the `raw_survey` table to avoid redundant loading.
- Loading the data into the database if necessary.
- Verifying the data load with a sample query.

## Step 1: Import Libraries and Set Up Database Connection
We import the required Python libraries and establish a connection to the PostgreSQL database using credentials stored in a `.env` file for security.

In [1]:
# Import libraries
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'dev_survey_insights')
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Create the database engine
engine = create_engine(DATABASE_URL)

## Step 2: Create Database and Table

We run the `setup.py` script to ensure the `dev_survey_insights` database and `raw_survey` table are created using SQLAlchemy.

In [2]:
# Run setup.py to create the database and table
%run ../scripts/setup.py

Base de datos 'dev_survey_insights' creada exitosamente.
Tabla 'raw_survey' creada o verificada exitosamente.


## Step 3: Load CSV Data into DataFrame

The raw survey data is loaded from the CSV file into a Pandas DataFrame for processing.

In [3]:
# Define the CSV file path
csv_path = '../data/survey_results_public.csv'

# Load the CSV data into a DataFrame
df = pd.read_csv(csv_path)
print(f"Data loaded successfully. Rows: {len(df)}, Columns: {len(df.columns)}")

Data loaded successfully. Rows: 98855, Columns: 129


C:\Users\Furas\AppData\Local\Temp\ipykernel_14100\3362206947.py:5: DtypeWarning: Columns (8,12,13,14,15,16,50,51,52,53,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


## Step 4: Check for Existing Data in Database

To prevent redundant loading and potential crashes, we check if the `raw_survey` table already contains data. If it does, we skip the loading step.

In [4]:
# Check if data is already loaded in the raw_survey table
query = "SELECT COUNT(*) FROM raw_survey"
count = pd.read_sql(query, engine).iloc[0, 0]

if count > 0:
    print("Data already exists in the 'raw_survey' table. Skipping the load step.")
else:
    # Load the data into the raw_survey table
    df.to_sql('raw_survey', engine, if_exists='replace', index=False)
    print("Data loaded into the 'raw_survey' table successfully.")

Data loaded into the 'raw_survey' table successfully.


## Step 5: Verify Data Loading

We query the first 5 rows of the `raw_survey` table to confirm that the data was loaded correctly.

In [5]:
# Verify the data by querying the first 5 rows
query = "SELECT * FROM raw_survey LIMIT 5;"
pd.read_sql(query, engine)

,Respondent,Hobby,OpenSource,Country,Student,Employment,FormalEducation,UndergradMajor,CompanySize,DevType,...,Exercise,Gender,SexualOrientation,EducationParents,RaceEthnicity,Age,Dependents,MilitaryUS,SurveyTooLong,SurveyEasy
0,1,Yes,No,Kenya,No,Employed part-time,"Bachelor’s degree (BA, BS, B.Eng., etc.)",Mathematics or statistics,20 to 99 employees,Full-stack developer,...,3 - 4 times per week,Male,Straight or heterosexual,"Bachelor’s degree (BA, BS, B.Eng., etc.)",Black or of African descent,25 - 34 years old,Yes,None,The survey was an appropriate length,Very easy
1,3,Yes,Yes,United Kingdom,No,Employed full-time,"Bachelor’s degree (BA, BS, B.Eng., etc.)","A natural science (ex. biology, chemistry, phy...","10,000 or more employees",Database administrator;DevOps specialist;Full-...,...,Daily or almost every day,Male,Straight or heterosexual,"Bachelor’s degree (BA, BS, B.Eng., etc.)",White or of European descent,35 - 44 years old,Yes,None,The survey was an appropriate length,Somewhat easy
2,4,Yes,Yes,United States,No,Employed full-time,Associate degree,"Computer science, computer engineering, or sof...",20 to 99 employees,Engineering manager;Full-stack developer,...,None,None,None,None,None,None,None,None,None,None
3,5,No,No,United States,No,Employed full-time,"Bachelor’s degree (BA, BS, B.Eng., etc.)","Computer science, computer engineering, or sof...",100 to 499 employees,Full-stack developer,...,I don't typically exercise,Male,Straight or heterosexual,Some college/university study without earning ...,White or of European descent,35 - 44 years old,No,No,The survey was an appropriate length,Somewhat easy
4,26,No,No,United States,"Yes, full-time",Employed part-time,"Bachelor’s degree (BA, BS, B.Eng., etc.)","Computer science, computer engineering, or sof...","1,000 to 4,999 employees",Student,...,3 - 4 times per week,None,None,None,None,None,None,None,None,None
